# Linear algebra basics

1. Pivoting demo
2. Computational complexity ($O(N^3)$)
3. "Don't invert": `solve` vs. `inv` and the downsides of inverting a matrix
4. Tridiagonal (banded) systems: the Thomas algorithm and `solve_banded`

In [ ]:
import timeit
import numpy as np
import scipy.linalg
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## 1. Why pivot?

Solve

$$
\begin{pmatrix} \epsilon & 1 \\ 1 & 1 \end{pmatrix}
\begin{pmatrix} x_0 \\ x_1 \end{pmatrix}
=
\begin{pmatrix} 1 \\ 2 \end{pmatrix},
$$

whose exact solution is $x_0 = 1/(1-\epsilon) \approx 1$, $x_1 = (1-2\epsilon)/(1-\epsilon) \approx 1$ for small $\epsilon$.

Here is Gaussian elimination plus back substitution, **without** pivoting. The first step divides by the pivot $A_{00}=\epsilon$.

In [ ]:
def gauss_nopivot(A, b):
    # Gaussian elimination + back substitution, no pivoting.
    A, b = A.astype(float), b.astype(float)    # work on copies
    n = len(b)
    for i in range(n):
        for k in range(i + 1, n):
            m = A[k, i] / A[i, i]
            A[k, i:] -= m * A[i, i:]
            b[k] -= m * b[i]
    x = np.zeros(n)
    for i in reversed(range(n)):
        x[i] = (b[i] - A[i, i + 1:] @ x[i + 1:]) / A[i, i]
    return x

**Partial pivoting** adds one step: before eliminating column $i$, swap row $i$ with the row (at or below $i$) that has the largest $|A_{ki}|$. Then every multiplier $m$ has $|m| \le 1$.

In [ ]:
def gauss_pivot(A, b):
    # Gaussian elimination + back substitution, with partial pivoting.
    A, b = A.astype(float), b.astype(float)
    n = len(b)
    for i in range(n):
        p = i + np.argmax(np.abs(A[i:, i]))     # row with the largest pivot
        A[[i, p]] = A[[p, i]]                   # swap rows i and p
        b[[i, p]] = b[[p, i]]
        for k in range(i + 1, n):
            m = A[k, i] / A[i, i]
            A[k, i:] -= m * A[i, i:]
            b[k] -= m * b[i]
    x = np.zeros(n)
    for i in reversed(range(n)):
        x[i] = (b[i] - A[i, i + 1:] @ x[i + 1:]) / A[i, i]
    return x

In [ ]:
b = np.array([1.0, 2.0])
methods = {"no pivoting": gauss_nopivot, "pivoting": gauss_pivot, "np.linalg.solve": np.linalg.solve}
print(f"{'eps':>6}   {'method':<16} {'x0':>16} {'x1':>16}")
for eps in [1e-5, 1e-10, 1e-15, 1e-20]:
    A = np.array([[eps, 1.0], [1.0, 1.0]])
    for name, solver in methods.items():
        x = solver(A, b)
        print(f"{eps:6.0e}   {name:<16} {x[0]:16.12f} {x[1]:16.12f}")
    print()

Without pivoting, the elimination step computes $1 - 1/\epsilon$, and the $1$ is lost to rounding once $1/\epsilon \gtrsim 10^{16}$. 

Back substitution then gives $x_0 = (1 - x_1)/\epsilon$: a difference of two nearly equal numbers, divided by a tiny one. 

At $\epsilon=10^{-10}$, $x_0$ has lost about half its digits; at $\epsilon=10^{-15}$ it is wrong in the 4th digit; at $\epsilon=10^{-20}$ it comes out as $0$.

`np.linalg.solve` calls LAPACK `_gesv`, which is LU decomposition **with** partial pivoting.

**Try this:** check that the matrix itself is not the problem: compute `np.linalg.cond(A)` for $\epsilon=10^{-20}$.

## 2. Computational complexity

Gaussian elimination and LU decomposition need $O(N^3)$ operations. Time `np.linalg.solve` for random $N\times N$ systems.

In [ ]:
Ns = np.array([100, 200, 400, 800, 1600, 3200])
times = []
for N in Ns:
    A = rng.normal(size=(N, N))
    b = rng.normal(size=N)
    times.append(min(timeit.repeat(lambda: np.linalg.solve(A, b), number=1, repeat=5)))
times = np.array(times)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(Ns, times, "o-", label="np.linalg.solve (LU, LAPACK)")
ax.loglog(Ns, times[-1] * (Ns / Ns[-1]) ** 3, "k--", label=r"$\propto N^{3}$")
ax.set(xlabel="$N$", ylabel="time [s]")
ax.legend()
plt.show()

At small $N$ the curve is flatter than $N^3$: the time is dominated by overhead (function calls, memory), and LAPACK uses multiple threads. The $N^3$ behavior takes over at large $N$.

**Try this:** extrapolate. How long would $N=10^5$ take? How much memory would the matrix need?

## 3. Don't invert: `solve` vs. `inv`

To solve $\mathbf{A}\mathbf{x}=\mathbf{b}$ you almost never need $\mathbf{A}^{-1}$. Computing `np.linalg.inv(A) @ b` costs about 3 times as much as `np.linalg.solve(A, b)`, and it is less accurate.

To see the accuracy difference, we need matrices that are hard to solve. We build $500\times500$ matrices with a chosen **condition number** $\kappa$ (the ratio of the largest to smallest singular value; more on this in the SVD lecture), and a known solution $\mathbf{x}_\text{true}$. We compare

- the **residual** $|\mathbf{A}\mathbf{x}-\mathbf{b}|/|\mathbf{b}|$: how well $\mathbf{x}$ satisfies the equation,
- the **error** $|\mathbf{x}-\mathbf{x}_\text{true}|/|\mathbf{x}_\text{true}|$: how close $\mathbf{x}$ is to the answer.

In [ ]:
n = 500
print(f"{'':7}   {' residual ':*^19}   {' error ':*^19}")
print(f"{'kappa':>7}   {'solve':>9} {'inv':>9}   {'solve':>9} {'inv':>9}")
for kappa in [1e2, 1e8, 1e12]:
    # A = U diag(s) V^T with random orthogonal U, V and singular values from 1 down to 1/kappa
    U, _ = np.linalg.qr(rng.normal(size=(n, n)))
    V, _ = np.linalg.qr(rng.normal(size=(n, n)))
    A = U @ np.diag(np.logspace(0, -np.log10(kappa), n)) @ V.T
    x_true = rng.normal(size=n)
    b = A @ x_true
    res, err = [], []
    for x in [np.linalg.solve(A, b), np.linalg.inv(A) @ b]:
        res.append(np.linalg.norm(A @ x - b) / np.linalg.norm(b))
        err.append(np.linalg.norm(x - x_true) / np.linalg.norm(x_true))
    print(f"{kappa:7.0e}   {res[0]:9.1e} {res[1]:9.1e}   {err[0]:9.1e} {err[1]:9.1e}")

- The **errors** grow like $\kappa \times 10^{-16}$ for both methods: that is a property of the matrix, and no algorithm can beat it. (`inv` is still somewhat worse.)
- The **residual** from `solve` stays at the $10^{-16}$ level for any $\kappa$ (LU with pivoting is *backward stable*), while the residual from `inv` grows with $\kappa$.

See N. Higham, *Accuracy and Stability of Numerical Algorithms*, ch. 14, or the [blog post by G. Gundersen](https://gregorygundersen.com/blog/2020/12/09/matrix-inversion/).

## 4. Tridiagonal systems

The finite-difference BVP from the lecture gives the matrix

$$
\mathbf{M} =
\begin{pmatrix}
2-c & -1 & & \\
-1 & 2-c & -1 & \\
 & \ddots & \ddots & \ddots \\
 & & -1 & 2-c
\end{pmatrix}.
$$

Almost all entries are zero:

In [ ]:
def bvp_matrix(N, c=0.01):
    return (np.diag(np.full(N, 2 - c)) + np.diag(np.full(N - 1, -1.0), 1)
            + np.diag(np.full(N - 1, -1.0), -1))

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.spy(bvp_matrix(20), markersize=7)
ax.set_title(r"nonzero entries of $\mathbf{M}$, $N=20$")
plt.show()

### Thomas algorithm

For a tridiagonal system, Gaussian elimination only has to touch one row below the diagonal and one element to the right of it, so it costs $O(N)$ instead of $O(N^3)$. Write the system as

$$
a_i x_{i-1} + b_i x_i + c_i x_{i+1} = d_i, \qquad a_0 = c_{N-1} = 0.
$$

Forward sweep: eliminate $a_i$, storing the modified $c_i'$ and $d_i'$. Then back-substitute. (No pivoting: this is safe when the matrix is diagonally dominant, $|b_i| \ge |a_i| + |c_i|$.)

In [ ]:
def thomas(a, b, c, d):
    # Solve a tridiagonal system. a: sub-diagonal (a[0] unused), b: diagonal,
    # c: super-diagonal (c[-1] unused), d: right-hand side. All of length N.
    N = len(d)
    cp = np.zeros(N)
    dp = np.zeros(N)
    cp[0] = c[0] / b[0]
    dp[0] = d[0] / b[0]
    for i in range(1, N):
        denom = b[i] - a[i] * cp[i - 1]
        cp[i] = c[i] / denom
        dp[i] = (d[i] - a[i] * dp[i - 1]) / denom
    x = np.zeros(N)
    x[-1] = dp[-1]
    for i in reversed(range(N - 1)):
        x[i] = dp[i] - cp[i] * x[i + 1]
    return x

N, c = 1000, 0.01
a = np.full(N, -1.0)
b = np.full(N, 2 - c)
cc = np.full(N, -1.0)
d = np.ones(N)
x_thomas = thomas(a, b, cc, d)
x_dense = np.linalg.solve(bvp_matrix(N, c), d)
print("relative difference, thomas vs. dense solve:",
      np.max(np.abs(x_thomas - x_dense)) / np.max(np.abs(x_dense)))

### `scipy.linalg.solve_banded`

Scipy's `solve_banded` method solves a system of linear equations for a "banded" matrix: besides the diagonal, the matrix has $l$ additional "lower diagonals" and $u$ additional "upper diagonals". This allows for both efficient storage and computation. 

First, take a minute to read https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_banded.html.

In brief, `solve_banded((l, u), ab, d)` takes the matrix in a compact "banded" storage: `ab` has one row per diagonal (here 3 rows of length $N$) instead of $N^2$ entries. The following cell compares its speed with a regular, dense solve method:

In [ ]:
print(f"{'N':>6} {'dense solve [s]':>16} {'solve_banded [s]':>17}")
for N in [100, 1000, 3000]:
    d = np.ones(N)
    bands = np.vstack([np.full(N, -1.0), np.full(N, 2 - c), np.full(N, -1.0)])
    M = bvp_matrix(N, c)
    t_dense = min(timeit.repeat(lambda: np.linalg.solve(M, d), number=1, repeat=3))
    t_band = min(timeit.repeat(lambda: scipy.linalg.solve_banded((1, 1), bands, d), number=1, repeat=3))
    print(f"{N:6d} {t_dense:16.1e} {t_band:17.1e}")

**Going further:**

- Time the pure-Python `thomas` function for the same values of $N$. Is it $O(N)$? How does it compare to `solve_banded` (compiled Fortran)? Could you speed it up with `numba`?
- `bvp.ipynb` solves the same kind of problem with `scipy.sparse`. Compare its approach with `solve_banded`.